# Test Pre-trained GM12878 Model (155 experiments, 50kbp)

This notebook evaluates the pre-trained GM12878 subcompartment model included in `TECSAS/share/models/`.

**Requirements:**
- Pre-trained weights: `TECSAS/share/models/bv_GM12878_155.pt` (included in repository)
- Training info checkpoint: `training_info_set_155.pt` (~900MB, download separately)

The training info checkpoint contains pre-computed train/test data splits. If unavailable, you can regenerate data from ENCODE using `data_process` (see `train_and_predict_HistMod_example.ipynb`).

In [ ]:
import os, sys
import numpy as np
import torch
from torch import Tensor
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# Import TECSAS
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'TECSAS'))
import TECSAS

In [ ]:
# Paths
weights_path = os.path.join(os.path.dirname(os.getcwd()), 'TECSAS', 'share', 'models')
checkpoint_path = './'  # Path to training_info_set_155.pt

# Model hyperparameters (must match pre-trained weights)
n_neighbors = 14
n_predict = 3
NEXP = 155
nbatches = 4000

emsize = 128
d_hid = 64
nlayers = 2
nhead = 8
dropout = 0.01
nfeatures = NEXP * (2 * n_neighbors + 1)
ostates = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Initialize model and load pre-trained weights
model = TECSAS.TECSAS(n_predict, emsize, nhead, d_hid, nlayers, nfeatures, ostates, dropout).to(device)

params = sum(np.prod(p.size()) for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {params:,}')

# Load weights (strip 'module.' prefix from DataParallel keys)
state = torch.load(os.path.join(weights_path, 'bv_GM12878_155.pt'), map_location='cpu', weights_only=True)
model.load_state_dict({'.'.join(k.split('.')[1:]): v for k, v in state.items()})
model.eval()
print('Model loaded successfully.')

In [ ]:
# Load test data from training info checkpoint
checkpoint = torch.load(os.path.join(checkpoint_path, 'training_info_set_155.pt'), map_location='cpu', weights_only=False)
train_data = checkpoint['train_data']
test_data = checkpoint['test_data']
ntest_loci = checkpoint['ntest_loci']

print(f"Checkpoint epoch: {checkpoint['epoch']}, best val loss: {checkpoint['best_val_loss']:.4f}")
print(f'Train samples: {len(train_data):,}, Test samples: {len(test_data):,}')

In [ ]:
# Evaluate on test set
def get_batch_test(source, i, n_predict, ndxs=None):
    data = source[i*bptt:(i+1)*bptt, 2*(n_predict-1)+1:][:, :, np.newaxis]
    target = source[i*bptt:(i+1)*bptt, :2*(n_predict-1)+1]
    indexes = ndxs[i*bptt:(i+1)*bptt]
    return data.to(device), target.to(device), indexes

bptt = len(train_data) // nbatches
nbatches_eval = len(test_data) // bptt

predictions, ground_truth = [], []
with torch.no_grad():
    for batch in range(nbatches_eval):
        data, targets, _ = get_batch_test(test_data, batch, n_predict=n_predict, ndxs=ntest_loci)
        pred = model(data, None)[0].argmax(dim=-1)[:, n_predict-1].cpu()
        predictions.append(pred)
        ground_truth.append(targets[:, n_predict-1].cpu())

predictions = np.concatenate(predictions)
ground_truth = np.concatenate(ground_truth)
print(f'Evaluated {len(predictions):,} test samples.')

In [ ]:
# Overall and per-class accuracy
labels = ['A1', 'A2', 'B1', 'B2', 'B3']
overall_acc = np.sum(predictions == ground_truth) / len(predictions)
print(f'Overall Accuracy: {overall_acc:.4f}')
print()
for i, name in enumerate(labels):
    mask = ground_truth == i
    acc = np.sum(predictions[mask] == ground_truth[mask]) / mask.sum()
    print(f'  {name}: {acc:.4f} ({mask.sum():,} samples)')

In [ ]:
# 5-class confusion matrix
cm = confusion_matrix(ground_truth, predictions, normalize='true')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        color = 'white' if cm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm[i,j]:.2f}', ha='center', va='center', color=color, fontsize=12)
ax.set_xticks(range(5)); ax.set_xticklabels(labels)
ax.set_yticks(range(5)); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'GM12878 Subcompartments (155 exp, 50kbp) — Acc: {overall_acc:.1%}')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# A/B compartment accuracy
ab_pred = predictions > 1  # B compartment: B1, B2, B3
ab_true = ground_truth > 1
ab_acc = np.sum(ab_pred == ab_true) / len(predictions)
print(f'A/B Accuracy: {ab_acc:.4f}')

cm_ab = confusion_matrix(ab_true, ab_pred, normalize='true')
ab_labels = ['A (A1+A2)', 'B (B1+B2+B3)']

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(cm_ab, cmap='Blues', vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        color = 'white' if cm_ab[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_ab[i,j]:.3f}', ha='center', va='center', color=color, fontsize=14)
ax.set_xticks(range(2)); ax.set_xticklabels(ab_labels)
ax.set_yticks(range(2)); ax.set_yticklabels(ab_labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'A/B Compartments — Acc: {ab_acc:.1%}')
plt.tight_layout()
plt.show()